# 面试问题：MoE Expert Parallel 的路由、容量与负载均衡怎样实现？

可直接复述的回答：MoE Router 为每个 token 产生 expert logits，再选择 Top-k 并归一化 gate。Expert Parallel 把专家分布到设备，dispatch 本质是按目的 rank 的 All-to-All。每个 expert 有容量，热点会造成 drop、排队或最慢 rank 拖尾。只看平均负载不够，要观察最热 expert、token drop 和通信矩阵。辅助负载损失或 loss-free bias 可以让路由更均衡，但过强会损伤专业化。溢出 token 应有第二专家、共享专家或 dense fallback。真实实现还需固定 token 顺序并在 combine 时恢复。

后续实验使用可读的小型业务数据验证关键判断。所有数值都标记为教学实验，不代表真实 GPU、线上流量或基础模型泛化结果。


## 1. 真实案例：客服 Token 与专家路由输入预览

十二个 token 来自账单、物流和退货请求，包含可读文本和三个专家 logits。前六个账单 token 形成热点，用于观察 capacity overflow；logits 是离线教学数据。


In [1]:
import numpy as np  # 使用 NumPy 实现 router softmax 和负载统计。
experts03 = ["billing", "delivery", "returns"]  # 定义三个客服专家。
token_rows03 = [  # 构造十二个带业务语义的 token 路由记录。
    ("发票", [3.2, 0.4, 0.2]),  # 强账单 token。
    ("扣款", [3.0, 0.5, 0.1]),  # 强账单 token。
    ("账单", [2.8, 0.6, 0.3]),  # 强账单 token。
    ("税号", [2.7, 0.4, 0.5]),  # 强账单 token。
    ("退款金额", [2.5, 0.8, 1.8]),  # 账单与退货交叉 token。
    ("支付失败", [2.6, 1.5, 0.3]),  # 账单与配送交叉 token。
    ("物流", [0.3, 3.1, 0.4]),  # 强配送 token。
    ("签收", [0.2, 2.9, 0.5]),  # 强配送 token。
    ("地址", [0.5, 2.5, 0.7]),  # 配送 token。
    ("退货", [0.4, 0.6, 3.0]),  # 强退货 token。
    ("破损", [0.6, 1.0, 2.7]),  # 退货 token。
    ("换货", [0.8, 0.7, 2.6]),  # 退货 token。
]  # 完成具有热点分布的 token batch。
logits03 = np.array([row03[1] for row03 in token_rows03], dtype=np.float64)  # 组成 Router logits 矩阵。
probabilities03 = np.exp(logits03 - logits03.max(axis=1, keepdims=True))  # 计算稳定未归一化 gate。
probabilities03 = probabilities03 / probabilities03.sum(axis=1, keepdims=True)  # 对每个 token 归一化专家概率。
print("教学实验输入：token | billing/delivery/returns gate")  # 输出输入预览表头。
for index03, row03 in enumerate(token_rows03):  # 逐 token 展示路由概率。
    print(row03[0], np.round(probabilities03[index03], 3).tolist())  # 输出可读 token 和 gate。


教学实验输入：token | billing/delivery/returns gate
发票 [0.9, 0.055, 0.045]
扣款 [0.879, 0.072, 0.048]
账单 [0.838, 0.093, 0.069]
税号 [0.826, 0.083, 0.091]
退款金额 [0.595, 0.109, 0.296]
支付失败 [0.698, 0.232, 0.07]
物流 [0.054, 0.887, 0.06]
签收 [0.058, 0.864, 0.078]
地址 [0.104, 0.769, 0.127]
退货 [0.064, 0.078, 0.858]
破损 [0.094, 0.14, 0.766]
换货 [0.126, 0.114, 0.761]


## 2. Baseline（基线）：Top-1 且超容量直接 Drop

每个 expert 容量设为 4。Top-1 会把六个账单 token 都发给 billing，后两个被丢弃；即使总容量等于 token 数，也会因分布不均浪费其他专家容量。


In [2]:
capacity03 = 4  # 设置每个 expert 的固定 token 容量。
top1_choices03 = np.argmax(probabilities03, axis=1)  # 获取每个 token 的最高概率专家。
baseline_load03 = np.bincount(top1_choices03, minlength=len(experts03))  # 统计 Top-1 专家负载。
baseline_dispatch03 = []  # 收集容量限制后的 dispatch 结果。
used03 = [0] * len(experts03)  # 初始化每个专家已接收 token 数。
for token_id03, expert_id03 in enumerate(top1_choices03.tolist()):  # 按原始 token 顺序执行 dispatch。
    if used03[expert_id03] < capacity03:  # 检查首选专家是否还有容量。
        used03[expert_id03] += 1  # 占用一个专家容量槽。
        baseline_dispatch03.append((token_rows03[token_id03][0], experts03[expert_id03], "accepted"))  # 记录成功 dispatch。
    else:  # 处理热点专家容量溢出。
        baseline_dispatch03.append((token_rows03[token_id03][0], experts03[expert_id03], "dropped"))  # 直接丢弃溢出 token。
baseline_drops03 = sum(row03[2] == "dropped" for row03 in baseline_dispatch03)  # 统计被丢弃 token 数。
print("Top-1基线：专家原始负载", dict(zip(experts03, baseline_load03.tolist())), "capacity", capacity03)  # 展示热点负载。
for row03 in baseline_dispatch03:  # 逐 token 展示容量结果。
    print(row03)  # 输出首选 expert 与 drop 状态。


Top-1基线：专家原始负载 {'billing': 6, 'delivery': 3, 'returns': 3} capacity 4
('发票', 'billing', 'accepted')
('扣款', 'billing', 'accepted')
('账单', 'billing', 'accepted')
('税号', 'billing', 'accepted')
('退款金额', 'billing', 'dropped')
('支付失败', 'billing', 'dropped')
('物流', 'delivery', 'accepted')
('签收', 'delivery', 'accepted')
('地址', 'delivery', 'accepted')
('退货', 'returns', 'accepted')
('破损', 'returns', 'accepted')
('换货', 'returns', 'accepted')


## 3. 核心实现：Top-2 容量感知 Dispatch 与负载指标

先按最高 gate 置信度处理 token；首选 expert 满时尝试第二专家，并记录 All-to-All 目的 rank。仍无容量时进入共享 dense fallback，而不是静默丢失。


In [3]:
top2_choices03 = np.argsort(-probabilities03, axis=1)[:, :2]  # 获取每个 token 的前两个专家。
confidence_order03 = np.argsort(-probabilities03.max(axis=1))  # 让高置信 token 优先占用专家容量。
core_dispatch03 = [None] * len(token_rows03)  # 按原始 token 位置保存最终 dispatch。
core_used03 = [0] * len(experts03)  # 初始化核心路由专家负载。
fallback03 = []  # 收集两个专家都满的 token。
for token_id03 in confidence_order03.tolist():  # 按 gate 置信度执行容量分配。
    assigned03 = False  # 初始化当前 token 分配状态。
    for rank03, expert_id03 in enumerate(top2_choices03[token_id03].tolist(), start=1):  # 依次尝试首选和次选专家。
        if core_used03[expert_id03] < capacity03:  # 检查候选专家剩余容量。
            core_used03[expert_id03] += 1  # 占用候选专家容量。
            core_dispatch03[token_id03] = (token_rows03[token_id03][0], experts03[expert_id03], f"top{rank03}")  # 保存分配专家和候选排名。
            assigned03 = True  # 标记当前 token 已分配。
            break  # 停止尝试其他专家。
    if not assigned03:  # 处理两个专家都无容量的 token。
        fallback03.append(token_rows03[token_id03][0])  # 将 token 送入共享 dense fallback。
        core_dispatch03[token_id03] = (token_rows03[token_id03][0], "shared_dense", "fallback")  # 保存可审计 fallback 路由。
importance03 = probabilities03.mean(axis=0)  # 计算每个专家平均 gate 重要性。
load_fraction03 = baseline_load03 / len(token_rows03)  # 计算离散 Top-1 负载比例。
auxiliary_loss03 = float(len(experts03) * np.sum(importance03 * load_fraction03))  # 计算常见 importance-load 辅助损失。
print("核心Dispatch：token | expert | route_rank")  # 输出容量感知 dispatch 表头。
for row03 in core_dispatch03:  # 逐 token 展示恢复原序后的路由。
    print(row03)  # 输出首选、次选或 fallback。
print("负载中间量", {"importance": np.round(importance03, 3).tolist(), "top1_load_fraction": np.round(load_fraction03, 3).tolist(), "aux_loss": round(auxiliary_loss03, 4)})  # 展示负载均衡统计。


核心Dispatch：token | expert | route_rank
('发票', 'billing', 'top1')
('扣款', 'billing', 'top1')
('账单', 'billing', 'top1')
('税号', 'billing', 'top1')
('退款金额', 'returns', 'top2')
('支付失败', 'delivery', 'top2')
('物流', 'delivery', 'top1')
('签收', 'delivery', 'top1')
('地址', 'delivery', 'top1')
('退货', 'returns', 'top1')
('破损', 'returns', 'top1')
('换货', 'returns', 'top1')
负载中间量 {'importance': [0.436, 0.291, 0.272], 'top1_load_fraction': [0.5, 0.25, 0.25], 'aux_loss': 1.0773}


## 4. 结果表与结果解读

Top-2 容量路由利用空闲专家并避免 drop，但部分交叉 token 被送到次选专家，可能牺牲专业化质量。真实训练需要同时比较任务 loss、专家负载和通信尾延迟。


In [4]:
core_drops03 = len(fallback03)  # 把共享 fallback 数量作为未进入稀疏专家的 token 数。
imbalance_baseline03 = int(baseline_load03.max() - baseline_load03.min())  # 计算 Top-1 最大最小负载差。
imbalance_core03 = int(max(core_used03) - min(core_used03))  # 计算容量感知后的负载差。
print("方法 | expert负载 | 稀疏drop/fallback | 最大最小负载差")  # 输出两种路由对照表头。
print("top1_drop", baseline_load03.tolist(), baseline_drops03, imbalance_baseline03)  # 展示热点丢弃基线。
print("top2_capacity", core_used03, core_drops03, imbalance_core03)  # 展示容量感知路由。
print("结果解读：Top-2减少热点丢弃，但次选比例和任务质量必须一起监控")  # 解释均衡不是唯一优化目标。


方法 | expert负载 | 稀疏drop/fallback | 最大最小负载差
top1_drop [6, 3, 3] 2 3
top2_capacity [4, 4, 4] 0 0
结果解读：Top-2减少热点丢弃，但次选比例和任务质量必须一起监控


## 5. 失败案例与修正：只看平均负载遗漏最慢 Rank

平均每专家四个 token 看似完美，但如果 billing 和 delivery 位于同一 rank，通信目的地仍可能热点。修正是同时统计 expert 和 rank 级 dispatch 矩阵。


In [5]:
expert_to_rank03 = {"billing": 0, "delivery": 0, "returns": 1, "shared_dense": 1}  # 定义专家到两个设备 rank 的映射。
rank_load03 = [0, 0]  # 初始化每个目的 rank 的 token 数。
for _, expert03, _ in core_dispatch03:  # 遍历容量感知后的 dispatch。
    rank_load03[expert_to_rank03[expert03]] += 1  # 累加目的设备通信负载。
naive_average03 = len(token_rows03) / 2.0  # 计算只看平均 rank 负载的错误摘要。
rank_imbalance03 = max(rank_load03) - min(rank_load03)  # 计算真实最慢 rank 负载差。
print("失败行为：只报平均rank负载", naive_average03)  # 展示平均值掩盖拓扑热点。
print("修正行为：逐rank目的负载", rank_load03, "差值", rank_imbalance03)  # 展示通信层真实不均衡。


失败行为：只报平均rank负载 6.0
修正行为：逐rank目的负载 [8, 4] 差值 4


## 6. 生产边界与专家并行合同

真实 MoE 需要 fused router、token permutation、All-to-All、反向 combine 和跨 rank 容量协调。负载 bias 要缓慢更新并做离线质量门禁，不能为了均衡强迫错误专家。


In [6]:
moe_contract03 = {"experts": experts03, "top_k": 2, "capacity_factor": 1.0, "overflow": "shared_dense", "expert_to_rank": expert_to_rank03, "router_metric": ["aux_loss", "drop", "rank_p95"]}  # 定义专家并行运行合同。
print("MoE 路由制品", moe_contract03)  # 展示容量、fallback 和拓扑字段。
print("生产替换点：fused router、All-to-All、token反排列、loss-free bias和跨rank尾延迟监控")  # 说明顺序模拟的边界。


MoE 路由制品 {'experts': ['billing', 'delivery', 'returns'], 'top_k': 2, 'capacity_factor': 1.0, 'overflow': 'shared_dense', 'expert_to_rank': {'billing': 0, 'delivery': 0, 'returns': 1, 'shared_dense': 1}, 'router_metric': ['aux_loss', 'drop', 'rank_p95']}
生产替换点：fused router、All-to-All、token反排列、loss-free bias和跨rank尾延迟监控


## 7. 最小回归测试

断言保护热点、容量路由和 rank 级审计。


In [7]:
assert len(token_rows03) >= 5  # 保证案例包含足够多的业务 token。
assert baseline_drops03 > 0  # 保证 Top-1 热点容量失败可见。
assert core_drops03 < baseline_drops03  # 保证 Top-2 容量路由减少稀疏 drop。
assert imbalance_core03 < imbalance_baseline03  # 保证核心路由改善专家负载差。
assert sum(rank_load03) == len(token_rows03)  # 保证所有 token 都进入专家或显式 fallback。
print("最小回归测试通过：Top-k、容量、fallback和rank负载审计稳定")  # 显示 MoE 关键性质已验证。


最小回归测试通过：Top-k、容量、fallback和rank负载审计稳定
